# ⚾ Homework: Moneyball Part 2, Runs Allowed
In class we found that **on-base percentage** and **slugging percentage** drive **runs scored**. But the A's win games through **run differential** (runs scored − runs allowed), so preventing runs matters just as much.

In this homework you'll build a regression model for **runs allowed**.

Run the setup cells below first. They load the same data and model-report helper we used in class.

## 🧰 Setup: load our tools

In [1]:
import pandas as pd
from pathlib import Path

# statsmodels builds our regression models
import statsmodels.api as sm
from statsmodels.tools.eval_measures import rmse

# sklearn scores our models and splits data into training and testing sets
from sklearn import metrics
from sklearn.model_selection import train_test_split

## 🔎 Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Renames the columns: abbreviations are spelled out using a dictionary, and every label is made lower case with underscores for spaces (`RS` becomes `runs_scored`).
3. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
4. Creates **run differential**: `runs_scored − runs_allowed`. It's stored as `runs_diff`.

In [7]:
# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ['data/baseball.csv', '../data/baseball.csv'] if Path(p).exists())
df = pd.read_csv(data_path)

In [8]:
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()

In [9]:
# Time machine: keep only seasons before Moneyball (2002)
df = df[df['year'] < 2002].reset_index(drop=True)

In [10]:
# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

df.head()

,team,league,year,runs_scored,runs_allowed,wins,on_base_percentage,slugging_percentage,batting_average,playoffs,rank_season,rank_playoffs,games,opponent_obp,opponent_slg,runs_diff
0,ANA,AL,2001,691,730,75,0.327,0.405,0.261,0,NaN,NaN,162,0.331,0.412,-39
1,ARI,NL,2001,818,677,92,0.341,0.442,0.267,1,5.0,1.0,162,0.311,0.404,141
2,ATL,NL,2001,729,643,88,0.324,0.412,0.260,1,7.0,3.0,162,0.314,0.384,86
3,BAL,AL,2001,687,829,63,0.319,0.380,0.248,0,NaN,NaN,162,0.337,0.439,-142
4,BOS,AL,2001,772,745,82,0.334,0.439,0.266,0,NaN,NaN,161,0.329,0.393,27


#### Data Dictionary

| **Column**              | **Original** | **Description**                                          |
| ----------------------- | ------------ | -------------------------------------------------------- |
| **team**                | Team         | Name of the team                                         |
| **league**              | League       | Name of the league                                       |
| **year**                | Year         | Year of the season                                       |
| **runs_scored**         | RS           | Runs scored by the team                                  |
| **runs_allowed**        | RA           | Runs allowed by the team                                 |
| **wins**                | W            | Number of wins                                           |
| **on_base_percentage**  | OBP          | On-Base Percentage                                       |
| **slugging_percentage** | SLG          | Slugging Percentage                                      |
| **batting_average**     | BA           | Batting Average                                          |
| **playoffs**            | Playoffs     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **rank_season**         | RankSeason   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **rank_playoffs**       | RankPlayoffs | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **games**               | G            | Number of games played                                   |
| **opponent_obp**        | OOBP         | Opponent On-Base Percentage                              |
| **opponent_slg**        | OSLG         | Opponent Slugging Percentage                             |
| **runs_diff**           | (new)        | Runs scored minus runs allowed (`runs_scored - runs_allowed`) |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

## 🛠️ Helper function
**`print_model_report`** is the same helper we used in class. It prints the coefficients and p-values, then R², RMSE, and the average of the dependent variable. By default it scores the model on the data it was trained on. Pass in different `X` and `y` (like a test set) to score it on that data instead.

In [11]:
def print_model_report(model, X=None, y=None):
    # Coefficient and p-value for each variable
    report = pd.DataFrame({'coefficient': model.params, 'p_value': model.pvalues})
    print(report.round(4))
    print()

    # By default, score the model on the data it was trained on
    # (model.model.exog holds the X values; model.model.endog holds the actual y values)
    if X is None:
        X = model.model.exog
        y = model.model.endog

    # How well the model fits
    y_pred = model.predict(X)
    print('R-squared:', round(metrics.r2_score(y, y_pred), 4))
    print('RMSE:', round(rmse(y, y_pred), 2))
    print(f'Average {model.model.endog_names}:', round(y.mean(), 2))

---
## 🧤 Your task: How do we ALLOW fewer runs?
Winning is about **run differential**, so preventing runs counts just as much as scoring them. Let's flip the question around and use the **opponents'** hitting stats against our pitchers:

* **Dependent variable:** `runs_allowed`
* **Try predictors:** `opponent_obp` (opponents' on-base %) and `opponent_slg` (opponents' slugging %)

⚠️ Check for nulls first (`df.info()`): `opponent_obp` and `opponent_slg` are **missing for most of the older seasons**. Drop those rows first (e.g. `df.dropna(subset=['opponent_obp', 'opponent_slg'])`), or the model will error out.

**Your turn:** build and evaluate this model by following the steps below. Then decide: do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model from class (R² ≈ 0.93 using `on_base_percentage` and `slugging_percentage`)?

### 1. Check for nulls and drop missing rows

In [18]:
# Check data types and non-null counts
print(df.info())

# Check total rows and columns
print("Rows before dropping nulls:", len(df))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   team                 902 non-null    object 
 1   league               902 non-null    object 
 2   year                 902 non-null    int64  
 3   runs_scored          902 non-null    int64  
 4   runs_allowed         902 non-null    int64  
 5   wins                 902 non-null    int64  
 6   on_base_percentage   902 non-null    float64
 7   slugging_percentage  902 non-null    float64
 8   batting_average      902 non-null    float64
 9   playoffs             902 non-null    int64  
 10  rank_season          154 non-null    float64
 11  rank_playoffs        154 non-null    float64
 12  games                902 non-null    int64  
 13  opponent_obp         90 non-null     float64
 14  opponent_slg         90 non-null     float64
 15  runs_diff            902 non-null    int

In [19]:
# Check for nulls, then drop the rows that are missing opponent_obp or opponent_slg
print(df[['opponent_obp', 'opponent_slg']].isnull().sum())

opponent_obp    812
opponent_slg    812
dtype: int64


In [20]:
df = df.dropna(subset = ['opponent_obp', 'opponent_slg']).reset_index(drop = True)
print(f"Remaining rows: {len(df)}")

Remaining rows: 90


### 2. Define X and y
Add a constant to `X` (`sm.add_constant`) so the model can learn an intercept.

In [21]:
# Define X (the predictors, plus a constant) and y (runs_allowed)
X = df[['opponent_obp', 'opponent_slg']]
X = sm.add_constant(X)

y = df['runs_allowed']

### 3. Build the model on all the data
Fit the model with `sm.OLS(y, X).fit()`, then print the results with `print_model_report`.

In [22]:
# Build the model on all the data and print the model report
model_all = sm.OLS(y,X).fit()
print_model_report(model_all)

              coefficient  p_value
const           -837.3779      0.0
opponent_obp    2913.5995      0.0
opponent_slg    1514.2860      0.0

R-squared: 0.9073
RMSE: 25.24
Average runs_allowed: 809.57


### 4. Evaluate on a train/test split
Split the data with `train_test_split(X, y, test_size=0.2, random_state=42)` so everyone gets the same numbers. Fit a model on the training data, then score it on the testing data with `print_model_report(model, X_test, y_test)`.

In [24]:
# Split the data, build the model on the training data, and score it on the testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_split = sm.OLS(y_train, X_train).fit()

print_model_report(model_split, X_test, y_test)

              coefficient  p_value
const           -812.7359      0.0
opponent_obp    2899.6840      0.0
opponent_slg    1470.0496      0.0

R-squared: 0.851
RMSE: 31.05
Average runs_allowed: 816.89


### 5. Interpret your results
✍️ **Answer here:** Do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model? How does the testing score compare with the score on all the data?

**Do the signs make sense?**

Yes. Both opponent_obp (~2900) and opponent_slg (~1470) have positive coefficients and p-values of 0.000. It makes complete sense: if opponents get on base more or hit for extra bases, our pitchers give up more runs. Also, opponent_obp's weight is basically double that of opponent_slg, which backs up the Moneyball idea that keeping guys off base matters way more than just limiting extra-base power.   

**How does the $R^2$ compare to the runs-scored model?**

It is very comparable. The runs-scored model from class had an $R^2 \approx 0.93$, and this full runs-allowed model hits an $R^2$ of about 0.91. So just like our hitters' OBP and SLG explain almost all our scoring, the opponents' OBP and SLG explain over 90% of the runs we give up.   

**Full data vs. Train/Test split:**

The test set takes a slight, totally normal dip: $R^2$ goes from 0.91 down to 0.85, and RMSE bumps up from ~25 to 31 runs. Still, an 0.85 $R^2$ on unseen test data is really solid and shows the model isn't overfitting - it's off by only about 31 runs over an entire 162-game season, which is around 0.2 runs a game against an average of ~817. 